# Xplore: **network**

This notebook analyses a PyPSA-$X$ network at any of the 7 network-building stages:

`base_network` · `simplify_network` · `cluster_network` · `add_electricity` ·
`prepare_network` · `prepare_sector_network` · `solve_sector_network`

**How it works**

- Set the `rule` you want to analyse in section *Parameters* (plus the `clusters/opts/...` wildcards).
- Sections are auto-detected from the loaded network (`functions.detect_sections`): only the
  ones with data for that state run.
- Map boundaries are read from `params.yaml`, selected
  with the `spatial_domain` tag (`'ES'`, `'EU'`, ...).

In [ ]:
######################################## Parameters

### Rule whose network you want to analyse (uncomment one of the following):
# rule = 'base_network'
# rule = 'simplify_network' 
# rule = 'cluster_network'
# rule = 'add_electricity'
# rule = 'prepare_network'
# rule = 'prepare_sector_network'
rule = 'solve_sector_network'


### Run
name = 'spain_off'
prefix = ''

### Scenario
clusters = 'adm'
opts = ''
sector_opts = ''
horizon = '2030'

### Spatial domain tag for the map boundaries. It must match a
### `boundaries_offshore_<tag>` entry in params.yaml (e.g. 'ES', 'EU').
spatial_domain = 'ES' 

In [ ]:
##### Import packages
import pypsa
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import sys


##### Import local functions
sys.path.append(os.path.abspath(os.path.join("..")))
import functions as xp


##### Read params.yaml
params = xp.read_params("../params.yaml")


##### Map boundaries for the selected spatial domain (from params.yaml)
boundaries = params[f'boundaries_offshore_{spatial_domain}']


##### Ignore warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

Load the network, its region files and auto-detect the available sections.

In [ ]:
##### Resolve the network source for the selected rule
src = xp.resolve_network_source(
    rule, clusters=clusters, opts=opts, sector_opts=sector_opts, horizon=horizon
)
print(f"rule       : {rule}")
print(f"file       : {src['filename']}")
print(f"location   : {src['location']}")
print(f"region_tag : {src['region_tag'] or '(none)'}")


##### Load the network
n = xp.load_network(
    params,
    src['filename'],
    prefix=prefix,
    name=name,
    location=src['location'],
)

##### Region files (onshore / offshore) matching this rule
gdf_regions_onshore, gdf_regions_offshore = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=src['region_tag'] or None,
)

##### Auto-detect which sections make sense for this network state
sections = xp.detect_sections(n)

n

Sections available for this network state:

In [ ]:
pd.Series(sections, name='available').to_frame()

## Network map

Plot the network framed by the `spatial_domain` boundaries from `params.yaml`.

In [ ]:
#################### Parameters
line_widths = 1 * n.lines.s_nom / 1e3
link_widths = 1 * n.links.p_nom / 1e3


#################### Figure
xp.plot_network_map(
    n,
    gdf_regions_onshore,
    gdf_regions_offshore,
    params,
    boundaries,
    line_widths=line_widths,
    link_widths=link_widths,
)

### Variable: `n.buses`

In [ ]:
n.buses.head()

### Variable: `n.carriers`

In [ ]:
n.carriers.head()

### Variable: `n.generators`

In [ ]:
if sections['generators']:
    gg = n.generators
    display(gg.head())
else:
    xp.notify_skipped('n.generators')

#### Summary

Aggregated and potential capacity per carrier.

In [ ]:
if sections['generators']:
    display(xp.summary_generators_by_carrier(n))
else:
    xp.notify_skipped('n.generators (summary)')

#### Summary by resource class

Solar and wind carriers may have classes. Table + bar plot of a feature per bus and class.

In [ ]:
if sections['generators']:
    #################### Parameters
    carrier = 'onwind'
    feature = 'p_nom'          # or 'p_nom_max'
    width = 0.3                # bar width

    gg = n.generators
    if ('wind' in carrier or 'solar' in carrier):
        gg_filtered = gg[gg['carrier'] == carrier][['bus', 'p_nom', 'p_nom_max']]
        gg_filtered['resource_class'] = gg_filtered.index.to_series().str.split(" ").str[-2]
        gg_filtered[['p_nom', 'p_nom_max']] = gg_filtered[['p_nom', 'p_nom_max']] / 1000
        gg_pivot = gg_filtered.pivot(index="bus", columns="resource_class", values=feature)

        display(
            gg_pivot.style
            .set_caption(f"{feature} [GW] for {carrier}, by bus and class.")
            .format("{:.2f}")
            .background_gradient(cmap="Blues")
        )

        #################### Bar plot
        df_sorted = gg_filtered.sort_values(["bus", "resource_class"])
        buses = df_sorted["bus"].unique()
        resources = df_sorted["resource_class"].unique()
        x = np.arange(len(buses))
        offsets = np.linspace(-width / 2.5, width / 2.5, len(resources))

        fig, ax = plt.subplots(figsize=(12, 8))
        colors = {"p_nom_max": "#cfe2f3", "p_nom": "#08306b"}
        labels_added = set()
        for i, res in enumerate(resources):
            df_res = df_sorted[df_sorted["resource_class"] == res]
            pos = x + offsets[i]
            for metric in ["p_nom_max", "p_nom"]:
                label = metric if metric not in labels_added else None
                ax.bar(pos, df_res[metric], width=width / len(resources),
                       color=colors[metric], alpha=0.7, label=label)
                labels_added.add(metric)
        ax.set_xticks(x)
        ax.set_xticklabels(buses, fontsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.set_xlabel("Bus", fontsize=16)
        ax.set_ylabel("Capacity [GW]", fontsize=16)
        ax.tick_params(axis="y", labelsize=14)
        ax.legend(fontsize=14)
        plt.tight_layout()
    else:
        print(f'Carrier {carrier} is not solar or wind type.. does it contain classes?')
else:
    xp.notify_skipped('n.generators (by resource class)')

#### Maps

Plot a feature of a generation carrier at each region.

In [ ]:
if sections['generators']:
    #################### Parameters
    carrier = 'onwind'
    resource_class = 'all'     # for solar/onwind: class number, or 'all'
    feature = 'p_nom'          # area | p_nom | p_nom_density | p_nom_max | p_nom_max_density | p_nom_max_ratio

    params_local = {'vmin': '', 'vmax': ''}

    #################### Figure
    fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
    gdf_regions = gdf_regions_offshore if 'off' in carrier else gdf_regions_onshore
    xp.map_add_features(ax, params['map_add_features'])
    xp.map_network_generators(
        carrier, n, feature, ax, gdf_regions, resource_class,
        params['map_network_generators'], params_local,
    )
else:
    xp.notify_skipped('n.generators (maps)')

#### Costs

Capital cost of generators for a selection of carriers.

In [ ]:
if sections['generators']:
    #################### Parameters
    carrier_list = ['onwind', 'solar', 'offwind-float']

    gg = n.generators
    fig, ax = plt.subplots(figsize=[10, 5])

    carrier_all = gg['carrier'].unique().tolist()
    carrier_excluded = [c for c in carrier_all if c not in carrier_list]
    print(f'Carriers ommitted: {carrier_excluded}')

    present = [c for c in carrier_list if c in carrier_all]
    tech_colors = n.carriers['color']
    if present:
        maximo = 1.05 * gg.loc[gg['carrier'].isin(present), 'capital_cost'].max()
        bins = np.arange(0, maximo, 1000)
        for carrier in present:
            df = gg[gg['carrier'] == carrier]
            if df['capital_cost'].round(2).nunique() == 1:
                valor = df['capital_cost'].unique()[0]
                ax.axvline(x=valor, label=carrier, color=tech_colors.get(carrier))
                print(f'Capital cost for {carrier} is: {valor:.2f} EUR/MW·year')
            else:
                ax.hist(df['capital_cost'], bins=bins, edgecolor='none',
                        color=tech_colors.get(carrier), label=carrier, alpha=1)
                print(f'Average capital cost for {carrier} is: {df["capital_cost"].mean():.2f} EUR/MW·year')
        ax.set_title('capital cost')
        ax.set_xlabel('EUR/(MW·year)')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.5)
    else:
        print('None of the selected carriers is present in this network.')
else:
    xp.notify_skipped('n.generators (costs)')

### Variable: `n.generators_t[p_max_pu]`

In [ ]:
if sections['pmaxpu']:
    ggt_pmaxpu = n.generators_t['p_max_pu']
    display(ggt_pmaxpu.head())
else:
    xp.notify_skipped("n.generators_t['p_max_pu']")

#### Time series

In [ ]:
if sections['pmaxpu']:
    #################### Parameters
    carrier = 'onwind'
    resource_class = 'all'
    start = '2013-03-01'
    end = '2013-04-01'

    ggt_pmaxpu = n.generators_t['p_max_pu']
    fig, ax = plt.subplots(figsize=[10, 4])
    if ('wind' in carrier or 'solar' in carrier) and resource_class != 'all':
        ggt_pmaxpu.loc[start:end].filter(like=f'{resource_class} {carrier}').plot(
            ax=ax, alpha=.7, legend=False, linewidth=.5)
    else:
        ggt_pmaxpu.loc[start:end].filter(like=carrier).plot(
            ax=ax, alpha=.7, legend=False, linewidth=.5)
    ax.grid(True, linestyle='--', alpha=0.25)
    ax.set_ylabel('')
else:
    xp.notify_skipped("n.generators_t['p_max_pu'] (time series)")

#### Summary by resource class

Capacity factor (CF) per bus and class (table + bar).

In [ ]:
if sections['pmaxpu']:
    #################### Parameters
    carrier = 'onwind'
    width = 0.3

    ggt_pmaxpu = n.generators_t['p_max_pu']
    if ('wind' in carrier or 'solar' in carrier):
        ggt_f = ggt_pmaxpu.filter(like=carrier, axis=1).mean().to_frame(name='CF')
        split_index = ggt_f.index.to_series().str.rsplit(' ', n=2, expand=True)
        ggt_f['bus'] = split_index[0].values
        ggt_f['resource_class'] = split_index[1].astype(int)
        ggt_pivot = ggt_f.pivot(index="bus", columns="resource_class", values='CF')
        display(
            ggt_pivot.style.set_caption(f"CF for {carrier}, by bus and class.")
            .format("{:.3f}").background_gradient(cmap="Blues")
        )

        df_sorted = ggt_f.sort_values(["bus", "resource_class"])
        buses = df_sorted["bus"].unique()
        resources = df_sorted["resource_class"].unique()
        x = np.arange(len(buses))
        offsets = np.linspace(-width / 2.5, width / 2.5, len(resources))
        fig, ax = plt.subplots(figsize=(12, 8))
        labels_added = set()
        for i, res in enumerate(resources):
            df_res = df_sorted[df_sorted["resource_class"] == res]
            label = 'CF' if 'CF' not in labels_added else None
            ax.bar(x + offsets[i], df_res['CF'], width=width / len(resources),
                   color="#bc91d8", alpha=0.7, label=label)
            labels_added.add('CF')
        ax.set_xticks(x)
        ax.set_xticklabels(buses, fontsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.set_xlabel("Bus", fontsize=16)
        ax.set_ylabel("CF", fontsize=16)
        ax.legend(fontsize=14)
        plt.tight_layout()
    else:
        print(f'Carrier {carrier} is not solar or wind type.. does it contain classes?')
else:
    xp.notify_skipped("n.generators_t['p_max_pu'] (by class)")

#### Maps

In [ ]:
if sections['pmaxpu']:
    #################### Parameters
    carrier = 'onwind'
    resource_class = 0         # note: 'all' is not valid here
    feature = 'CF'

    params_local = {'vmin': '', 'vmax': ''}
    fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
    gdf_regions = gdf_regions_offshore if 'off' in carrier else gdf_regions_onshore
    xp.map_add_features(ax, params['map_add_features'])
    xp.map_network_generatorst_pmaxpu(
        carrier, n, feature, ax, gdf_regions,
        params['map_network_generatorst_pmaxpu'], params_local, resource_class,
    )
else:
    xp.notify_skipped("n.generators_t['p_max_pu'] (maps)")

### Variable: `n.generators_t[p]` (dispatch)

In [ ]:
if sections['solved']:
    ggt_p = n.generators_t['p']
    display(ggt_p.head())
else:
    xp.notify_skipped("n.generators_t['p'] (dispatch, solved network only)")

#### Time series

In [ ]:
if sections['solved']:
    #################### Parameters
    carrier = 'onwind'
    resource_class = 'all'
    start = '2013-03-01'
    end = '2013-04-01'

    ggt_p = n.generators_t['p']
    fig, ax = plt.subplots(figsize=[10, 4])
    if ('wind' in carrier or 'solar' in carrier) and resource_class != 'all':
        ggt_p.loc[start:end].filter(like=f'{resource_class} {carrier}').plot(
            ax=ax, alpha=.7, legend=False, linewidth=.5)
    else:
        ggt_p.loc[start:end].filter(like=carrier).plot(
            ax=ax, alpha=.7, legend=False, linewidth=.5)
    ax.grid(True, linestyle='--', alpha=0.25)
    ax.set_ylabel('MW')
else:
    xp.notify_skipped("n.generators_t['p'] (time series)")

#### Summary by resource class

Annual energy production (AEP) per bus and class (table + bar).

In [ ]:
if sections['solved']:
    #################### Parameters
    carrier = 'onwind'
    width = 0.3

    ggt_p = n.generators_t['p']
    if ('wind' in carrier or 'solar' in carrier):
        ggt_f = ggt_p.filter(like=carrier, axis=1).multiply(
            xp.snapshot_hours(n), axis=0).sum().to_frame(name='AEP')
        split_index = ggt_f.index.to_series().str.rsplit(' ', n=2, expand=True)
        ggt_f['bus'] = split_index[0].values
        ggt_f['resource_class'] = split_index[1].astype(int)
        ggt_f[['AEP']] = ggt_f[['AEP']] / 1e6
        ggt_pivot = ggt_f.pivot(index="bus", columns="resource_class", values='AEP')
        display(
            ggt_pivot.style.set_caption(f"AEP [TWh] for {carrier}, by bus and class.")
            .format("{:.3f}").background_gradient(cmap="Blues")
        )

        df_sorted = ggt_f.sort_values(["bus", "resource_class"])
        buses = df_sorted["bus"].unique()
        resources = df_sorted["resource_class"].unique()
        x = np.arange(len(buses))
        offsets = np.linspace(-width / 2.5, width / 2.5, len(resources))
        fig, ax = plt.subplots(figsize=(12, 8))
        labels_added = set()
        for i, res in enumerate(resources):
            df_res = df_sorted[df_sorted["resource_class"] == res]
            label = 'AEP' if 'AEP' not in labels_added else None
            ax.bar(x + offsets[i], df_res['AEP'], width=width / len(resources),
                   color="#93c543", alpha=0.7, label=label)
            labels_added.add('AEP')
        ax.set_xticks(x)
        ax.set_xticklabels(buses, fontsize=14)
        ax.grid(axis="y", linestyle="--", alpha=0.6)
        ax.set_xlabel("Bus", fontsize=16)
        ax.set_ylabel("AEP [TWh]", fontsize=16)
        ax.legend(fontsize=14)
        plt.tight_layout()
    else:
        print(f'Carrier {carrier} is not solar or wind type.. does it contain classes?')
else:
    xp.notify_skipped("n.generators_t['p'] (by class)")

#### Maps

In [ ]:
if sections['solved']:
    #################### Parameters
    carrier = 'onwind'
    resource_class = 'all'
    feature = 'AEP'

    params_local = {'vmin': '', 'vmax': ''}
    fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
    gdf_regions = gdf_regions_offshore if 'off' in carrier else gdf_regions_onshore
    xp.map_add_features(ax, params['map_add_features'])
    xp.map_network_generatorst_p(
        carrier, n, feature, ax, gdf_regions, resource_class,
        params['map_network_generatorst_p'], params_local,
    )
else:
    xp.notify_skipped("n.generators_t['p'] (maps)")

### Variable: `n.global_constraints`

In [ ]:
if sections['global_constraints']:
    display(n.global_constraints)
else:
    xp.notify_skipped('n.global_constraints')

### Variable: `n.lines`

In [ ]:
if sections['lines']:
    ln = n.lines
    display(ln.head())
else:
    xp.notify_skipped('n.lines')

How is the distribution of line lengths?

In [ ]:
if sections['lines'] and 'length' in n.lines:
    xp.plot_length_hist(n.lines['length'], bins=50, xlabel='km')
else:
    xp.notify_skipped('n.lines (length histogram)')

Relationship between line capital cost and length.

In [ ]:
if sections['lines'] and {'length', 'capital_cost'}.issubset(n.lines.columns):
    xp.plot_cost_vs_length(n.lines, ylabel='EUR/MW', unit='EUR/(MW·km)')
else:
    xp.notify_skipped('n.lines (cost vs length)')

### Variable: `n.links`

In [ ]:
if sections['links']:
    lk = n.links
    display(lk.head())
else:
    xp.notify_skipped('n.links')

Carriers associated with links.

In [ ]:
if sections['links']:
    display(n.links['carrier'].drop_duplicates().reset_index(drop=True))
else:
    xp.notify_skipped('n.links (carriers)')

#### Link type: DC

In [ ]:
if sections['links'] and (n.links['carrier'] == 'DC').any():
    lk_DC = n.links.loc[n.links['carrier'] == 'DC']
    display(lk_DC.head())
    if 'length' in lk_DC:
        xp.plot_length_hist(lk_DC['length'], bins=50, xlabel='km')
    if {'length', 'capital_cost'}.issubset(lk_DC.columns):
        xp.plot_cost_vs_length(lk_DC, ylabel='EUR', unit='EUR/km')
else:
    xp.notify_skipped('n.links (DC)')

#### Link types: maps

Plot a feature of a link carrier at each region (e.g. `H2 Electrolysis`, `battery charger`, `CCGT`, `OCGT`, ...).

In [ ]:
if sections['links']:
    #################### Parameters
    carrier = 'H2 Electrolysis'
    feature = 'p_nom_e'        # area | p_nom_e

    params_local = {'vmin': '', 'vmax': ''}
    if carrier in n.links['carrier'].unique():
        fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
        xp.map_add_features(ax, params['map_add_features'])
        xp.map_network_links(
            carrier, n, feature, ax, gdf_regions_onshore,
            params['map_network_links'], params_local,
        )
    else:
        print(f"Carrier '{carrier}' not present. Available: "
              f"{sorted(n.links['carrier'].unique())}")
else:
    xp.notify_skipped('n.links (maps)')

### Variable: `n.loads`

In [ ]:
if sections['loads']:
    display(n.loads.head())
    if 'carrier' in n.loads:
        display(n.loads.groupby('carrier').size().to_frame('count'))
else:
    xp.notify_skipped('n.loads')

### Variable: `n.loads_t[p_set]`

In [ ]:
if sections['loads_t']:
    lot_pset = n.loads_t['p_set']
    display(lot_pset.head())
else:
    xp.notify_skipped("n.loads_t['p_set']")

#### Time series

Optionally filter by a `load_key` (e.g. `'heating'`). Leave `None` for all loads. Note: the built-in `'electricity'` key is defined for pypsa-spain bus names.

In [ ]:
if sections['loads_t']:
    #################### Parameters
    load_key = None            # None | 'heating' | 'electricity'
    start = '2013-01-01'
    end = '2013-01-31'

    lot_pset = n.loads_t['p_set']
    ts_full = xp.filter_df_columns(lot_pset, filter_key=load_key) if load_key else lot_pset

    if ts_full.shape[1] == 0:
        print(f"No load columns match load_key={load_key!r}.")
    else:
        fig, ax = plt.subplots(figsize=[10, 4])
        ts_full.loc[start:end].plot(ax=ax, alpha=.4, legend=False, linewidth=.5)
        # Weight by snapshot duration [h] so energy is correct for any temporal resolution
        hours = xp.snapshot_hours(n)
        total_twh = ts_full.multiply(hours, axis=0).sum().sum() / 1e6
        ax.set_title(f"Load time series (total: {total_twh:.2f} TWh)")
        ax.grid(True, linestyle='--', alpha=1)
        ax.set_ylabel('MW')
else:
    xp.notify_skipped("n.loads_t['p_set'] (time series)")

#### Maps

In [ ]:
if sections['loads_t']:
    #################### Parameters
    feature = 'annual_load_density'   # area | annual_load | annual_load_density
    load_key = None                   # None | 'heating' | 'electricity'

    params_local = {'vmin': '', 'vmax': ''}
    fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
    xp.map_add_features(ax, params['map_add_features'])
    xp.map_network_loadst_pset(
        n, feature, ax, gdf_regions_onshore,
        params['map_network_loadst_pset'], params_local, load_key=load_key,
    )
else:
    xp.notify_skipped("n.loads_t['p_set'] (maps)")

### Variable: `n.storage_units`

In [ ]:
if sections['storage_units']:
    su = n.storage_units
    display(su.head())
else:
    xp.notify_skipped('n.storage_units')

#### Summary

In [ ]:
if sections['storage_units']:
    su = n.storage_units
    display(su.groupby('carrier').agg(
        Total_capacity=pd.NamedAgg(column='p_nom', aggfunc='sum'),
        Buses=pd.NamedAgg(column='p_nom', aggfunc='size'),
        Buses_zero_capacity=pd.NamedAgg(column='p_nom', aggfunc=lambda x: (x == 0).sum()),
        Buses_non_zero_capacity=pd.NamedAgg(column='p_nom', aggfunc=lambda x: (x != 0).sum()),
    ))
else:
    xp.notify_skipped('n.storage_units (summary)')

#### Maps

In [ ]:
if sections['storage_units']:
    #################### Parameters
    carrier = 'PHS'
    feature = 'max_hours'      # area | p_nom | p_nom_density | max_hours

    params_local = {'vmin': '', 'vmax': ''}
    if carrier in n.storage_units['carrier'].unique():
        fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
        xp.map_add_features(ax, params['map_add_features'])
        xp.map_network_storage_units(
            carrier, n, feature, ax, gdf_regions_onshore,
            params['map_network_storage_units'], params_local,
        )
    else:
        print(f"Carrier '{carrier}' not present. Available: "
              f"{sorted(n.storage_units['carrier'].unique())}")
else:
    xp.notify_skipped('n.storage_units (maps)')

#### Maximum hours

In [ ]:
if sections['storage_units']:
    su = n.storage_units
    fig, ax = plt.subplots(figsize=[8, 4])
    tech_colors = n.carriers['color']
    for carrier, group in su.groupby('carrier'):
        ax.scatter(group['p_nom'], group['max_hours'], label=carrier,
                   color=tech_colors.get(carrier), alpha=0.7)
    ax.set_xlabel('Installed capacity [MW]')
    ax.set_ylabel('Max. hours')
    ax.set_title('Storage units')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
else:
    xp.notify_skipped('n.storage_units (max hours)')

### Variable: `n.stores`

In [ ]:
if sections['stores']:
    st = n.stores
    display(st.head())
    display(st['carrier'].drop_duplicates().reset_index(drop=True))
else:
    xp.notify_skipped('n.stores')

#### Maps

In [ ]:
if sections['stores']:
    #################### Parameters
    carrier = 'battery'        # e.g. battery | H2 Store
    feature = 'e_nom_opt'      # area | e_nom_opt

    params_local = {'vmin': '', 'vmax': ''}
    if carrier in n.stores['carrier'].unique():
        fig, ax = plt.subplots(figsize=[12, 6], subplot_kw={'projection': ccrs.PlateCarree()})
        xp.map_add_features(ax, params['map_add_features'])
        xp.map_network_stores(
            carrier, n, feature, ax, gdf_regions_onshore,
            params['map_network_stores'], params_local,
        )
    else:
        print(f"Carrier '{carrier}' not present. Available: "
              f"{sorted(n.stores['carrier'].unique())}")
else:
    xp.notify_skipped('n.stores (maps)')

### Variable: `n.shapes`

In [ ]:
if sections['shapes']:
    sh = n.shapes
    display(sh.head())
    sh.plot()
else:
    xp.notify_skipped('n.shapes')

### Variable: `n.transformers`

In [ ]:
if sections['transformers']:
    display(n.transformers.head())
else:
    xp.notify_skipped('n.transformers')

## `regions_onshore`

In [ ]:
display(gdf_regions_onshore.head())
print(f"Number of onshore regions: {len(gdf_regions_onshore)}")
gdf_regions_onshore.plot()

## `regions_offshore`

In [ ]:
display(gdf_regions_offshore.head())
print(f"Number of offshore regions: {len(gdf_regions_offshore)}")
gdf_regions_offshore.plot()

# Analysis

The following analyses only apply to a **solved** network (`solve_sector_network`).

### Analysis: Capacity expansion

Summary of initial and optimal capacities. Capacity of links `CCGT`, `OCGT`, `H2 Fuel Cell` and `battery discharger` refers to electric capacity (input link capacity is multiplied by efficiency).

In [ ]:
if sections['solved']:
    df_capacities = xp.df_network_capacities(n)
    display(df_capacities)

    #################### Parameters
    carrier_list = ['solar', 'onwind', 'offwind-float', 'CCGT', 'DC', 'battery charger']

    present = [c for c in carrier_list if c in df_capacities.index]
    if present:
        fig, ax = plt.subplots(figsize=[10, 6])
        df_capacities.loc[present].plot(kind='bar', rot=45, ax=ax)
        ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)
    else:
        print('None of the selected carriers is present.')
else:
    xp.notify_skipped('Analysis: Capacity expansion')

### Analysis: Grid expansion

Where was grid expansion required? Plot the increase of line/link capacities.

In [ ]:
if sections['solved']:
    line_widths = 1 * (n.lines.s_nom_opt - n.lines.s_nom) / 1e3
    link_widths = 1 * (n.links.p_nom_opt - n.links.p_nom) / 1e3
    xp.plot_network_map(
        n, gdf_regions_onshore, gdf_regions_offshore, params, boundaries,
        line_widths=line_widths, link_widths=link_widths,
    )
else:
    xp.notify_skipped('Analysis: Grid expansion')

### Analysis: Grid congestion

Where are the bottlenecks in the grid from the optimal dispatch?

In [ ]:
if sections['solved']:
    #################### Parameters
    cong_criterion = ('quantile', 0.9)   # or ('mean', 'na')

    line_widths = 1 * (n.lines.s_nom_opt) / 1e3
    link_widths = 0 * (n.links.p_nom_opt) / 1e3   # WIP: link congestion
    lnt_p0 = n.lines_t['p0']

    #################### Line colors
    if cong_criterion[0] == 'mean':
        ln_result_criterion = lnt_p0.abs().mean() / n.lines['s_nom']
        title_cb = 'Mean of line CF'
    else:  # 'quantile'
        ln_result_criterion = lnt_p0.abs().quantile(cong_criterion[1]) / n.lines['s_nom']
        title_cb = f'Q{cong_criterion[1]} of line CF'

    norm = mcolors.Normalize(vmin=0, vmax=0.7)
    cmap = plt.cm.rainbow
    ln_colors = [cmap(norm(p)) for p in ln_result_criterion]

    #################### Figure
    fig, ax = plt.subplots(figsize=[12, 12], subplot_kw={'projection': ccrs.PlateCarree()})
    n.plot(ax=ax, line_widths=line_widths, link_widths=link_widths,
           line_colors=ln_colors, bus_sizes=params['bus_sizes'],
           bus_colors=params['bus_colors'], boundaries=boundaries)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical', shrink=0.55, pad=0.02)
    cbar.set_label(title_cb, fontsize=12)
    xp.map_add_region(ax, gdf_regions_onshore, params['map_add_region'])
    xp.map_add_features(ax, params['map_add_features'])
else:
    xp.notify_skipped('Analysis: Grid congestion')

### Analysis: Interconnections

Energy imports/exports through border links. Only for networks with interconnections enabled (border links named `*export*`/`*import*`, e.g. pypsa-spain).

In [ ]:
if sections['interconnections']:
    exports_t = n.links_t['p0'].filter(like='export')
    imports_t = -n.links_t['p1'].filter(like='import')

    # Weight by snapshot duration [h] so energy is correct for any temporal resolution
    hours = xp.snapshot_hours(n)
    import_bar = pd.DataFrame(imports_t.multiply(hours, axis=0).sum().div(1e6)).T
    export_bar = pd.DataFrame(exports_t.multiply(hours, axis=0).sum().div(1e6)).T
    import_bar.index = ['Imports']
    export_bar.index = ['Exports']
    combined = pd.concat([import_bar, export_bar])

    ic_color_dict = xp.get_ic_colors(combined.columns)
    colors = [ic_color_dict[col] for col in combined.columns]

    fig, ax = plt.subplots(figsize=[6, 6])
    combined.plot(kind='bar', stacked=True, color=colors, ax=ax, width=0.7)
    ax.set_ylabel("TWh")
    ax.set_xlabel("")
    ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)
    ax.legend(title='Border links', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.tight_layout()
    plt.show()
else:
    xp.notify_skipped('Analysis: Interconnections')

Interconnection power time series (exports positive, imports negative).

In [ ]:
if sections['interconnections']:
    #################### Parameters
    start = '2023-03-01'
    end = '2023-03-15'

    exports_t = n.links_t['p0'].filter(like='export')
    imports_t = -n.links_t['p1'].filter(like='import')
    ic_color_dict = xp.get_ic_colors(list(exports_t.columns) + list(imports_t.columns))

    exports_plot = exports_t.loc[start:end].div(1e3).abs()
    imports_plot = -imports_t.loc[start:end].div(1e3).abs()
    export_colors = [ic_color_dict.get(col, '#808080') for col in exports_plot.columns]
    import_colors = [ic_color_dict.get(col, '#808080') for col in imports_plot.columns]

    fig, ax = plt.subplots(figsize=[10, 4])
    ax.stackplot(exports_plot.index, exports_plot.T.values,
                 labels=[f"{c} (export)" for c in exports_plot.columns],
                 colors=export_colors, alpha=0.85)
    ax.stackplot(imports_plot.index, imports_plot.T.values,
                 labels=[f"{c} (import)" for c in imports_plot.columns],
                 colors=import_colors, alpha=0.45)
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.7)
    ax.set_ylabel('Power [GW]')
    ax.set_xlabel('')
    ax.grid(True, linestyle='--', linewidth=0.5, color='black', alpha=0.25)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=9)
    plt.tight_layout()
else:
    xp.notify_skipped('Analysis: Interconnections (time series)')